# Pipeline Preditivo de Manutencao Industrial

Projeto de Ciencia de Dados aplicado a predicao de falhas mecanicas em equipamentos industriais a partir de leituras de sensores de temperatura, rotacao, torque e desgaste de ferramenta.

## Fase 1 - Exploracao de Dados (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

df = pd.read_csv("data/manutencao_preditiva.csv")
print("Dimensoes da base (linhas, colunas):", df.shape)
print("\n Estrutura e tipos de dados:")
df.info()

In [ ]:
df.describe().T

In [ ]:
colunas_sensores = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, col in enumerate(colunas_sensores):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[idx], color="#2b5c8f", bins=30)
    axes[idx].set_title(f"Distribuicao de {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Frequencia")

axes[5].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
contagem_alvo = df["falha_maquina"].value_counts()
proporcao_alvo = df["falha_maquina"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 5))
barras = sns.barplot(
    x=contagem_alvo.index.astype(str),
    y=contagem_alvo.values,
    hue=contagem_alvo.index.astype(str),
    palette=["#2b5c8f", "#c93b2b"],
    legend=False,
    ax=ax
)

for idx, valor in enumerate(contagem_alvo.values):
    pct = proporcao_alvo.iloc[idx]
    ax.text(idx, valor + 150, f"{valor} ({pct:.2f}%)", ha="center", fontweight="bold")

ax.set_title("Desbalanceamento da Variavel Alvo (falha_maquina)")
ax.set_xlabel("Estado da Maquina (0 = Operacao Normal, 1 = Falha)")
ax.set_ylabel("Quantidade de Registros")
ax.set_ylim(0, 11000)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
matriz_corr = df[colunas_sensores + ["falha_maquina"]].corr(method="pearson")
sns.heatmap(
    matriz_corr,
    annot=True,
    fmt=".4f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Mapa de Calor - Correlacao Linear de Pearson")
plt.tight_layout()
plt.show()

### Analise Interpretativa da Exploracao de Dados

A inspecao inicial da base `data/manutencao_preditiva.csv` mostra exatamente 10.000 registros e 14 colunas. A leitura cruzada com o documento tecnico do Departamento de Engenharia define a funcao de cada atributo:
- As colunas `udi` (1 a 10.000) e `id_produto` (10.000 codigos exclusivos) funcionam apenas como chaves identificadoras de registro e serie, sem valor preditivo generalizavel.
- As colunas `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf` e `falha_rnf` registram o motivo especifico da quebra apos a ocorrencia da falha. Conforme especificado pela engenharia, essas cinco variaveis sao registros de historico pos-evento e devem ser removidas da matriz preditora `X` para evitar vazamento direto da resposta.
- A coluna `tipo` classifica a qualidade do equipamento em tres categorias: `L` com 6.000 registros (60,00%), `M` com 2.997 registros (29,97%) e `H` com 1.003 registros (10,03%).

O `.info()` e o `.describe()` revelam que exatamente 4 colunas continuas possuem 9.500 registros preenchidos e 500 valores ausentes (`NaN`): `temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`. Ja `desgaste_ferramenta_min` possui os 10.000 registros completos, variando entre 0 e 253 minutos (media de 107,95 minutos e mediana de 108,00 minutos).

A analise dos tres graficos evidencia tres comportamentos centrais para o pipeline:
1. **Distribuicao das variaveis preditoras:** As variaveis `temperatura_ar_k` (media de 300,00 K e mediana de 300,10 K), `temperatura_processo_k` (media de 310,00 K e mediana de 310,10 K) e `torque_nm` (media de 39,97 Nm e mediana de 40,10 Nm) apresentam distribuicoes simetricas proximas a normalidade. Em contrapartida, `velocidade_rotacao_rpm` apresenta forte assimetria positiva (cauda longa a direita chegando ao maximo de 2.886,0 RPM), deslocando a media (1.539,25 RPM) acima da mediana (1.504,00 RPM).
2. **Desbalanceamento severo da classe alvo:** A variavel `falha_maquina` contabiliza 9.661 registros na classe 0 (96,61% de operacao normal) contra apenas 339 registros na classe 1 (3,39% de falha mecanica). Treinar classificadores diretamente nessa proporcao enviesaria o modelo para prever sempre a classe majoritaria.
3. **Correlacoes lineares de Pearson:** Existe correlacao positiva forte de `0.8766` entre `temperatura_ar_k` e `temperatura_processo_k`, refletindo o acoplamento termico entre o ambiente da fabrica e o processo de usinagem. Observa-se tambem correlacao negativa forte de `-0.8754` entre `velocidade_rotacao_rpm` e `torque_nm`, comportamento fisico esperado em motores eletricos onde o aumento da carga de torque reduz a rotacao. Em relacao ao alvo `falha_maquina`, as maiores correlacoes individuais ocorrem com `torque_nm` (`0.1889`) e `desgaste_ferramenta_min` (`0.1054`).